# Assemble by match and team identity

## Goal and setup
Build one aligned model dataset from keyed features and a selected label. This
example uses the pinned Premier League 2024/25 prepared exports and the existing
`misc314_py314` kernel. No temporal split, imputation or fitting is performed.

Feature evaluation retains its earlier-finished-kickoff availability proxy when
no release column is supplied; labels are actual outcomes. Assembly applies no
additional time filter. See the [practical guide](../docs/analytics/datasets.md)
and [full contract](../docs/analytics/datasets_reference.md).

In [1]:
from pathlib import Path
import pandas as pd
from xdiyo_analytics.data import load_seasons, select_stats
from xdiyo_analytics.histories import build_team_history
from xdiyo_analytics.features import Stat, ForAgainst, IsHome, Lag, RollingMean, evaluate_features
from xdiyo_analytics.labels import TeamValue, MatchTotal, Outcome, BetOption, create_labels
from xdiyo_analytics.datasets import ModelDataset, assemble_dataset

root = Path('C:/Users/luisi/Documents/Programming/Python/xDiyo')
data = load_seasons(
    root / 'data/xDiyo_data', ['24_25'], leagues='Premier_League',
    tables=['matches', 'statistics'],
    record_dir=root / 'experiment/initial_population/selections', verify_hashes=True,
)
history = build_team_history(select_stats(data, stats=[
    (period, 'Match overview', 'cornerKicks') for period in ['ALL', '1ST', '2ND']
]))
corners = Stat('ALL', 'Match overview', 'cornerKicks')
periods = Stat(None, 'Match overview', 'cornerKicks')


In [2]:
feature_definitions = {
    'home_context': IsHome(),
    'recent_corners': Lag(ForAgainst(corners, side='both')),
    'mean_corners': RollingMean(corners, 3),
}
features = evaluate_features(history, feature_definitions, keyed=True)
labels = create_labels(history, {
    'team_corners': TeamValue(periods),
    'match_corners': MatchTotal(periods),
    'away_outcome': Outcome(perspective='away'),
    'over_10': BetOption(MatchTotal(corners), 'over', line=10, push_value=0.5),
})
features.index.names


FrozenList(['source_league', 'source_season', 'competition_id', 'season_id', 'event_id', 'team_id', 'side'])

## Steps: shuffle safely, then choose a layout
`keyed=True` keeps identity fields in a named MultiIndex. Explicit key columns
also work. Labels determine output order; anonymous index or row position is
never the join key. Match layout places home features before away features.

In [3]:
shuffled_features = features.sample(frac=1, random_state=193)
match_data = assemble_dataset(
    shuffled_features, labels['match_corners'], layout='match',
)
explicit_feature_columns = shuffled_features.reset_index()
same_match_data = assemble_dataset(
    explicit_feature_columns, labels['match_corners'], layout='match',
)
pd.testing.assert_frame_equal(match_data.X, same_match_data.X)

print(match_data.X.shape, match_data.y.shape)
match_data.X.head(4)

(380, 8) (380, 3)


,home::home_context,home::recent_corners::team::ALL::Match overview::cornerKicks::value,home::recent_corners::opponent::ALL::Match overview::cornerKicks::value,home::mean_corners,away::home_context,away::recent_corners::team::ALL::Match overview::cornerKicks::value,away::recent_corners::opponent::ALL::Match overview::cornerKicks::value,away::mean_corners
0,1.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN
1,1.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN
2,1.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN
3,1.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN


In [4]:
team_data = assemble_dataset(features, labels['team_corners'], layout='team_match')
away_data = assemble_dataset(features, labels['away_outcome'], layout='match')
assert team_data.X.shape == (760, 4)
assert match_data.X.shape == (380, 8)
assert away_data.target_perspective == 'away'
match_data.X.head(4)


,home::home_context,home::recent_corners::team::ALL::Match overview::cornerKicks::value,home::recent_corners::opponent::ALL::Match overview::cornerKicks::value,home::mean_corners,away::home_context,away::recent_corners::team::ALL::Match overview::cornerKicks::value,away::recent_corners::opponent::ALL::Match overview::cornerKicks::value,away::mean_corners
0,1.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN
1,1.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN
2,1.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN
3,1.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN


## Checks: groups and settlement stay explicit
Home/away prefixes identify the focal feature row; for/against identifies the
historical statistic within that row. An away-outcome target keeps its away
perspective without swapping feature blocks. Group tuples identify matches;
they are not ranker group sizes. Settlement belongs in metadata.

In [5]:
option_data = assemble_dataset(features, labels['over_10'], layout='match')
paired = assemble_dataset(features, labels['team_corners'], layout='team_match', drop_missing_targets=True)
assert paired.groups.value_counts().eq(2).all()
assert all(frame.index.equals(option_data.X.index) for frame in [option_data.y, option_data.metadata])
assert not any(name.startswith('settlement::') for name in option_data.X)
print(f'{match_data.groups.nunique()} match groups; {len(team_data.y)} team rows')
pd.concat([
    option_data.metadata[['event_id', 'home_id', 'away_id', 'settlement::over_10']],
    option_data.y,
], axis=1).head(6)

380 match groups; 760 team rows


,event_id,home_id,away_id,settlement::over_10,over_10
0,12436870,35,43,win,1.0
1,12436871,32,44,win,1.0
2,12436872,42,3,push,0.5
3,12436873,48,30,loss,0.0
4,12436874,39,45,win,1.0
5,12436875,14,60,loss,0.0


## Takeaways and next steps
The example produces a match feature matrix of 380 rows and eight columns, or
760 team rows and four columns. Three supplied statistic periods stay separate
target columns. All outputs use a fresh shared RangeIndex and exact IDs remain
in metadata. Optional missing-target filtering removes whole matches, preserving
team pairs; missing feature values remain available for later handling.

Temporal split, preprocessing/model adapters and reports are subsequent layers.
See the [coverage checklist](../docs/analytics/datasets_documentation_checklist.md)
for independently checked identities, filtering and preservation.